# Week 3, day 3 (afternoon) — Worksheet 04: What COPY INTO actually does   (runs locally)

`3_Stage_tables.sql` loads both files with a **transformational COPY** — a COPY
whose source is a `SELECT`, so it can reshape the data on the way in:

```sql
COPY INTO STG_Sales (TRANS_ID, ..., SHIP_COST, BATCH_ID)
FROM (
    SELECT $1, $2, ..., $14,
           SPLIT_PART(METADATA$FILENAME, '/', -1)
    FROM @"SALES_STAGE"/data_loading_lab/csv_files/sales_2013_01_01.csv
)
FILE_FORMAT = (
    TYPE                         = 'CSV'
    SKIP_HEADER                  = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    DATE_FORMAT                  = 'MM/DD/YYYY'
);
```

Four settings and a SELECT. This worksheet rebuilds each one in pandas, on the
same file, so you can see what each clause is *for* — and what the table looks
like when one of them is missing.

**Question 4 is supposed to raise an error.**

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 04 — What COPY INTO does. Run this once.
import pandas as pd

PRODUCTS = "data/products_2013_01_01.csv"
SALES = "data/sales_2013_01_01.csv"

# The 14 positions the COPY selects, in order, named as STG_Sales declares them.
STG_SALES_COLS = [
    "TRANS_ID", "PROD_KEY", "STORE_KEY", "TRANS_DT", "TRANS_TIME",
    "PRIORITY", "SALES_QTY", "SALES_PRICE", "SALES_AMT",
    "DISCOUNT", "SALES_COST", "SALES_MGRN", "SHIP_MODE", "SHIP_COST",
]

print("rebuilding the COPY INTO in", len(STG_SALES_COLS), "positions + BATCH_ID")

PART A — SKIP_HEADER and positional selection

### Question 1

`SKIP_HEADER = 1` plus `SELECT $1 ... $14` means: throw the header away, then take fields by position. Reproduce that with `header=None`, `skiprows=1` and `names=STG_SALES_COLS`. Print the shape and the first two rows of `TRANS_ID`, `TRANS_DT` and `SHIP_MODE`.
> **NOTE:** the CSV header says `SHIPMODE`; you are naming that position `SHIP_MODE`, exactly as the COPY does. Positional loading lets you rename on arrival.

In [ ]:
############################
## Your Code Here
############################

### Question 2

Confirm nothing was lost. Count the data lines in the raw file with `open()` (all lines minus the header) and compare with `len(stg)`.

In [ ]:
############################
## Your Code Here
############################

### Question 3

Now do it wrong: load the same file with `header=None` but **without** `skiprows`. Print the first row and the dtype of `TRANS_ID`.
> **NOTE:** this is a COPY with `SKIP_HEADER` left off. Snowflake would reject the row rather than accept it as text, but the lesson is the same — one setting decides whether the header is data.

In [ ]:
############################
## Your Code Here
############################

### Question 4

That bad load emitted a `DtypeWarning` about mixed types. Count the actual Python types in `bad["SALES_AMT"]` with `.map(type).value_counts()`, print the last position holding a `str` and the first holding a `float`, then total the column. **The total is supposed to fail.**
> **NOTE:** the split is not where the bad row is. Look at the number and ask what else in pandas has that size.

In [ ]:
############################
## Your Code Here
############################

PART B — FIELD_OPTIONALLY_ENCLOSED_BY, and why products does not use it

### Question 5

The sales FILE_FORMAT sets `FIELD_OPTIONALLY_ENCLOSED_BY = '"'`; the products FILE_FORMAT does not. Print the first raw line of data from each file and explain the difference in one sentence.

In [ ]:
############################
## Your Code Here
############################

### Question 6

Strip the surviving quotes the way a transformational SELECT would — `TRIM($6, '\"')`. Apply `.str.strip('\"')` to `PRIORITY` and `SHIP_MODE`, then print the distinct values of each with `repr()` and the row count for `PRIORITY == "High"`.

In [ ]:
############################
## Your Code Here
############################

PART C — DATE_FORMAT

### Question 7

`TRANS_DT` is declared `DATE` in `STG_Sales`, and the FILE_FORMAT sets `DATE_FORMAT = 'MM/DD/YYYY'`. Parse it with `format="%m/%d/%Y"`, print the dtype and min/max. Then parse the same column with `format="%d/%m/%Y"` inside a try/except and print what happens.
> **NOTE:** `3/7/2010` is a valid date under both readings. That is exactly the problem.

In [ ]:
############################
## Your Code Here
############################

### Question 8

Find how many rows are ambiguous — where day and month are both 12 or less, so the string parses to a valid but *different* date under the other convention. Print the count and the percentage.

In [ ]:
############################
## Your Code Here
############################

PART D — BATCH_ID

### Question 9

The COPY appends a 15th column: `SPLIT_PART(METADATA$FILENAME, '/', -1)`. Given the staged path `data_loading_lab/csv_files/sales_2013_01_01.csv`, reproduce that in Python and add the result as `BATCH_ID`. Print the column list and the distinct `BATCH_ID` values.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Build the finished staging table: positional load, quotes stripped, `TRANS_DT` parsed as a date, `BATCH_ID` added. Then print the row count grouped by `BATCH_ID` — the same audit `4_Validate_stage_tables.sql` runs — and, beside it, the row count grouped by the *year of* `TRANS_DT`.

In [ ]:
############################
## Your Code Here
############################